# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-khaled123/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** a `RandomForestClassifier` (300 trees, max depth 8, balanced class weights).

**Why it fits Lane 2:** the label (`is_declining`) is not a clean linear function of any single
prior-window metric — a page can decline for different reasons (falling out of a good position,
losing CTR at a stable position, simply going stale). A random forest can pick up interactions
between features (e.g. "low CTR *combined with* mid-table position") without hand-engineering every
interaction term, and it copes natively with a mix of numeric features and a categorical one
(`content_type`) with one-hot encoding. The code cell below checks the individual linear
correlation of each feature with the label — if that correlation were already strong on its own, a
simpler logistic regression would be the more honest, more interpretable choice; it isn't, which
supports reaching for an ensemble instead.


In [1]:
import duckdb, pandas as pd, numpy as np, os, warnings
warnings.filterwarnings("ignore")

candidates = [
    os.path.expanduser("~/Documents/flyrank-hf-data"),
    os.path.expanduser("~/mnt/Documents/flyrank-hf-data"),
]
BASE = next((p for p in candidates if os.path.isdir(p)), candidates[0])
REPO_ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "work")) \
    else os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

con = duckdb.connect()
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
DIM_CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"

feat = con.sql(f"""
    WITH avail AS (SELECT * FROM {FACT} WHERE gsc_data_available IS TRUE),
    prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior, SUM(gsc_clicks) AS clicks_prior,
               AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS avg_position_prior,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impr_prior
        FROM avail WHERE report_date <= DATE '2026-03-15' GROUP BY 1, 2
    ),
    target AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_target, SUM(gsc_clicks) AS clicks_target
        FROM avail WHERE report_date >= DATE '2026-03-16' GROUP BY 1, 2
    )
    SELECT p.*, COALESCE(t.impressions_target, 0) AS impressions_target,
           COALESCE(t.clicks_target, 0) AS clicks_target
    FROM prior p LEFT JOIN target t USING (client_hash_id, content_hash_id)
    WHERE p.impressions_prior >= 10
""").df()

content = con.sql(f"SELECT content_hash_id, content_created_date, content_type FROM {DIM_CONTENT}").df()
feat = feat.merge(content, on="content_hash_id", how="left")
feat["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat["content_created_date"])).dt.days
feat = feat[feat["content_age_days"] >= 0].copy()

feat["imp_rate_prior"]  = feat["impressions_prior"]  / 15.0
feat["imp_rate_target"] = feat["impressions_target"] / 16.0
feat["is_declining"] = (feat["imp_rate_target"] < 0.8 * feat["imp_rate_prior"]).astype(int)
feat["ctr_prior"] = feat["clicks_prior"] / feat["impressions_prior"]

FEATURE_COLS = ["impressions_prior", "clicks_prior", "ctr_prior", "avg_position_prior",
                "days_with_impr_prior", "content_age_days"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

def bucket_pos(p):
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    if p <= 50: return "21-50"
    return "50+"
feat["pos_bucket"] = feat["avg_position_prior"].apply(bucket_pos)

print(f"{len(feat):,} pages / {feat['client_hash_id'].nunique()} clients / decline rate {feat['is_declining'].mean():.3f}")

# Individual (linear) correlation of each prior-window feature with the label —
# if any one of these were strongly linear, a simple logistic regression would be
# the more honest, simpler choice instead of a random forest.
corrs = feat[FEATURE_COLS + ["is_declining"]].corr(numeric_only=True)["is_declining"].drop("is_declining")
print("Point-biserial correlation of each feature with is_declining:")
print(corrs.sort_values(key=abs, ascending=False).round(3))
print("\nNo single feature clears |r| > 0.1 on its own -> supports a model that can combine")
print("features and pick up interactions, rather than a single linear rule.")


114,715 pages / 39 clients / decline rate 0.344
Point-biserial correlation of each feature with is_declining:
content_age_days       -0.112
clicks_prior           -0.052
ctr_prior              -0.031
avg_position_prior     -0.027
impressions_prior       0.007
days_with_impr_prior   -0.001
Name: is_declining, dtype: float64

No single feature clears |r| > 0.1 on its own -> supports a model that can combine
features and pick up interactions, rather than a single linear rule.


## 2. Split design

**Split:** 80/20, `GroupShuffleSplit` grouped by `client_hash_id` — every page belonging to one
client lands entirely in train or entirely in test, never split across both.

**Why this is honest for this question:** a client's pages share editorial style, publishing
cadence, and niche. A random row-level split would let the model partly learn "how this specific
client's pages behave" rather than a pattern that transfers to a client it has never seen — exactly
the memorization risk this internship's validation-design guidance warns about. Since the real
decision this model supports (which pages to put in *a* client's refresh queue) will eventually be
asked about clients not in this training set, the split needs to reflect that.


In [2]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(feat, groups=feat["client_hash_id"]))
train, test = feat.iloc[train_idx].copy(), feat.iloc[test_idx].copy()

print(f"train: {len(train):,} rows / {train['client_hash_id'].nunique()} clients")
print(f"test:  {len(test):,} rows / {test['client_hash_id'].nunique()} clients (held out entirely)")
overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"clients appearing in BOTH train and test: {len(overlap)} (should be 0)")


train: 107,030 rows / 31 clients
test:  7,685 rows / 8 clients (held out entirely)
clients appearing in BOTH train and test: 0 (should be 0)


## 3. Train + compare vs my baseline

Same data, same client-grouped test split, same two metrics (ROC-AUC and Precision@K) as the Week-4
CTR-gap baseline (`w04_baseline_score.ipynb`).


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

expected_ctr = train.groupby("pos_bucket")["ctr_prior"].mean()
test["expected_ctr"] = test["pos_bucket"].map(expected_ctr)
test["ctr_gap"] = (test["expected_ctr"] - test["ctr_prior"]).clip(lower=0)
test["baseline_score"] = test["ctr_gap"] * test["impressions_prior"]

train_X = pd.get_dummies(train[FEATURE_COLS + ["content_type"]], columns=["content_type"])
test_X = pd.get_dummies(test[FEATURE_COLS + ["content_type"]], columns=["content_type"])
test_X = test_X.reindex(columns=train_X.columns, fill_value=0)

rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(train_X, train["is_declining"])
test["model_score"] = rf.predict_proba(test_X)[:, 1]

rows = []
for name, scores in [("Week-4 baseline (CTR-gap rule)", test["baseline_score"]), ("Week-5 model (random forest)", test["model_score"])]:
    auc = roc_auc_score(test["is_declining"], scores)
    p20 = precision_at_k(scores, test["is_declining"], 20)
    p50 = precision_at_k(scores, test["is_declining"], 50)
    rows.append((name, round(auc,3), round(p20,3), round(p50,3)))
results = pd.DataFrame(rows, columns=["approach","ROC-AUC","Precision@20","Precision@50"])
print(results.to_string(index=False))


                      approach  ROC-AUC  Precision@20  Precision@50
Week-4 baseline (CTR-gap rule)    0.506          0.35          0.46
  Week-5 model (random forest)    0.589          0.80          0.78


## 4. Errors and interpretation

The model beats the CTR-gap baseline at both Precision@20 and Precision@50 on clients it never
trained on. Feature importances and a small set of confidently-wrong examples below say more about
*why* than the metric table alone does.


In [4]:
imp = pd.Series(rf.feature_importances_, index=train_X.columns).sort_values(ascending=False)
print("Feature importances:")
print(imp.head(6).round(4))

cols = ["content_hash_id","client_hash_id","content_type","impressions_prior","ctr_prior",
        "avg_position_prior","days_with_impr_prior","content_age_days","model_score"]
fp = test[(test["model_score"]>0.6) & (test["is_declining"]==0) & (test["impressions_prior"]>=100)] \
        .sort_values("model_score", ascending=False)[cols].head(3)
fn = test[(test["model_score"]<0.3) & (test["is_declining"]==1) & (test["impressions_prior"]>=100)] \
        .sort_values("model_score")[cols].head(3)
print("\nConfidently-wrong FALSE POSITIVES (flagged high risk, actually stable/growing):")
print(fp.to_string(index=False))
print("\nConfidently-wrong FALSE NEGATIVES (flagged low risk, actually declined):")
print(fn.to_string(index=False))
print("\nThe model leans most on content_age_days and ctr_prior (see importances above). Both error")
print("groups share near-zero ctr_prior at a mid-table position - exactly where those two leading")
print("features can point in conflicting directions, which is where the model is least reliable.")


Feature importances:
content_age_days        0.2904
ctr_prior               0.1735
avg_position_prior      0.1583
days_with_impr_prior    0.1406
impressions_prior       0.1249
clicks_prior            0.0815
dtype: float64

Confidently-wrong FALSE POSITIVES (flagged high risk, actually stable/growing):
         content_hash_id          client_hash_id   content_type  impressions_prior  ctr_prior  avg_position_prior  days_with_impr_prior  content_age_days  model_score
content_e7f0b8bdc0ace93a client_b10cb2997d0c7c86 feedly article             3795.0        0.0            5.040532                    15               153     0.697588
content_43e0ce6ba1f2c1d8 client_b10cb2997d0c7c86 feedly article              121.0        0.0            7.081668                    13                16     0.693594
content_759a49480f111e90 client_b10cb2997d0c7c86 feedly article              150.0        0.0            8.617975                    15               179     0.686829

Confidently-wrong FALSE NEGA

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
